# Proyecto Final Integrador (TEFI) — Minería de Datos 2026
## Clasificación Automática y Descubrimiento de Patrones de Calidad en Hojas de Tabaco

* **Institución:** IES N° 7 "Populorum Progressio - In.Te.La." — San Salvador de Jujuy  
* **Carrera:** Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial (3.er año)  
* **Cátedra:** Minería de Datos (2026) — Docente: Ing. Fernando Cabana  
* **Autor:** Lucas Victor Maciel Alancay Sarapura  
* **Metodología:** CRISP-DM (Unidades 1 a 6) sobre dataset real de visión por computadora (`tobacco-leaf-detection` v1 — Roboflow, 512 × 512 px)

## Paso 0 · Configuración del Entorno y Carga Interactiva del Dataset
Antes de iniciar el análisis, este bloque prepara las librerías y habilita la carga de tu archivo **`_annotations.csv`** (y, de forma opcional, un archivo **`.zip`** con tus imágenes de entrenamiento para extraer métricas de color y textura directamente sobre cada recorte).

In [1]:
import os
import io
import zipfile
import itertools
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedGroupKFold, cross_validate
)
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
)
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, mean_absolute_error, mean_squared_error,
    r2_score, silhouette_score, adjusted_rand_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

print("Entorno configurado correctamente. RANDOM_STATE =", RANDOM_STATE)

Entorno configurado correctamente. RANDOM_STATE = 42


In [2]:
# ==============================================================================
# CARGA INTERACTIVA DEL DATASET (CSV de Roboflow + ZIP opcional de imágenes)
# ==============================================================================
# Si ejecutás este cuaderno en Google Colab y todavía no subiste el archivo,
# se abrirá automáticamente el botón para seleccionar `_annotations.csv` (y opcionalmente un `.zip` con fotos).
# Cambiá FORZAR_SUBIDA = True si querés reemplazar un dataset cargado previamente.

FORZAR_SUBIDA = False
RUTA_CSV = "_annotations.csv"
CARPETA_IMAGENES = "imagenes_tabaco"

def cargar_datos_usuario(ruta_csv=RUTA_CSV, carpeta_img=CARPETA_IMAGENES, forzar=FORZAR_SUBIDA):
    try:
        from google.colab import files
        en_colab = True
    except ImportError:
        en_colab = False

    if en_colab and (forzar or not os.path.exists(ruta_csv)):
        print("Seleccioná tu archivo `_annotations.csv` (y opcionalmente un `.zip` con tus imágenes):")
        subidos = files.upload()
        for nombre, contenido in subidos.items():
            if nombre.lower().endswith(".csv"):
                with open(ruta_csv, "wb") as f:
                    f.write(contenido)
                print(f"-> CSV guardado como: {ruta_csv}")
            elif nombre.lower().endswith(".zip"):
                os.makedirs(carpeta_img, exist_ok=True)
                with zipfile.ZipFile(io.BytesIO(contenido)) as zf:
                    zf.extractall(carpeta_img)
                print(f"-> Imágenes extraídas en: {carpeta_img}/")

    if not os.path.exists(ruta_csv):
        raise FileNotFoundError(
            f"No se encontró '{ruta_csv}'. Subilo desde el panel izquierdo de Colab o ejecutá nuevamente esta celda."
        )

    df_cargado = pd.read_csv(ruta_csv)
    print(f"Dataset cargado exitosamente: {df_cargado.shape[0]} filas x {df_cargado.shape[1]} columnas.")
    return df_cargado

df_raw = cargar_datos_usuario()
display(df_raw.head())

Dataset cargado exitosamente: 711 filas x 8 columnas.


,filename,width,height,class,xmin,ymin,xmax,ymax
0,622049355_1809812529709566_1591487239658484514_n_jpg.rf.0b86a50fbcea7c3b49ac30fa989ed179.jpg,512,512,tobacco leaf,1,0,512,449
1,617518128_1915235752696599_36536970339096276_n_jpg.rf.569dfef432ea8135e6df53c251ce3f53.jpg,512,512,tobacco leaf,30,1,385,512
2,622976949_1258054016175912_8768215516095570699_n_jpg.rf.33b643d65c13068d7ba34fa87a52c1d6.jpg,512,512,tobacco leaf,127,0,512,511
3,625065897_817552418012862_5481226591888427787_n_jpg.rf.5f9be830d8d0df810cc5e5a80948bd4d.jpg,512,512,tobacco leaf,94,0,512,511
4,625266409_1283778333578720_4675737267050600593_n_jpg.rf.d569327b6e4efc7e8d47d3bdf01fe06b.jpg,512,512,tobacco leaf,0,126,511,510


---
## Unidad 1 · Comprensión del Negocio, Análisis Exploratorio (EDA) y Ética
1. **Problema de negocio:** Automatizar la detección de hojas de tabaco enfermas (`unhealthy tobacco leaf`) frente a hojas sanas (`tobacco leaf`) y cuantificar la cobertura foliar en imágenes de 512 × 512 px exportadas desde Roboflow (`tobacco-leaf-detection` v1).
2. **Criterio de éxito técnico:** Priorizar la sensibilidad (**Recall**) y el puntaje **F1** sobre la clase minoritaria (`unhealthy tobacco leaf`) para evitar que hojas deterioradas pasen sin alerta al acopio, además de auditar la fuga de datos provocada por el *data augmentation*.
3. **Marco ético (UNESCO, 2021):** Transparencia sobre el desbalance de clases, prevención de sesgos de encuadre (hojas recortadas contra los bordes de la cámara) y validación sin contaminación entre copias espejadas de una misma fotografía.

In [3]:
print("=== DIMENSIONES Y TIPOS DE DATOS ===")
print(f"Registros totales: {len(df_raw)} | Columnas: {df_raw.shape[1]}")
print(df_raw.dtypes)

print("\n=== VALORES NULOS POR COLUMNA ===")
print(df_raw.isnull().sum())

n_dup = df_raw.duplicated().sum()
n_imgs = df_raw["filename"].nunique()
base_imgs = df_raw["filename"].str.split("_jpg.rf.").str[0].nunique()

print(f"\nFilas duplicadas exactas: {n_dup}")
print(f"Imágenes únicas en CSV (con augmentation): {n_imgs}")
print(f"Fotografías originales base (sin augmentation): {base_imgs}")

print("\n=== DISTRIBUCIÓN ORIGINAL DE CLASES ===")
print(df_raw["class"].value_counts(dropna=False))

=== DIMENSIONES Y TIPOS DE DATOS ===
Registros totales: 711 | Columnas: 8
filename    object
width        int64
height       int64
class       object
xmin         int64
ymin         int64
xmax         int64
ymax         int64
dtype: object

=== VALORES NULOS POR COLUMNA ===
filename    0
width       0
height      0
class       0
xmin        0
ymin        0
xmax        0
ymax        0
dtype: int64

Filas duplicadas exactas: 1
Imágenes únicas en CSV (con augmentation): 232
Fotografías originales base (sin augmentation): 99

=== DISTRIBUCIÓN ORIGINAL DE CLASES ===
class
tobacco leaf              640
unhealthy tobacco leaf     63
Unlabeled                   5
Unhealthy                   3
Name: count, dtype: int64


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Conteo de clases crudas
vc = df_raw["class"].value_counts()
sns.barplot(x=vc.index, y=vc.values, ax=axes[0, 0], palette=["#2E8B57", "#C0392B", "#7F8C8D", "#E67E22"])
axes[0, 0].set_title("1. Distribución de Clases en Anotaciones Crudas")
axes[0, 0].set_ylabel("Cantidad de Cajas")
axes[0, 0].tick_params(axis="x", rotation=12, labelsize=9)
for i, v in enumerate(vc.values):
    axes[0, 0].text(i, v + 8, str(v), ha="center", fontweight="bold")

# 2. Hojas anotadas por imagen
cajas_img = df_raw.groupby("filename").size()
sns.histplot(cajas_img, bins=range(1, cajas_img.max() + 2), discrete=True, ax=axes[0, 1], color="#3465A4")
axes[0, 1].set_title("2. Cantidad de Hojas Anotadas por Imagen")
axes[0, 1].set_xlabel("Hojas por imagen")
axes[0, 1].set_ylabel("Cantidad de imágenes")

# 3. Distribución de ancho vs alto de las cajas
ancho_tmp = df_raw["xmax"] - df_raw["xmin"]
alto_tmp = df_raw["ymax"] - df_raw["ymin"]
sns.scatterplot(
    x=ancho_tmp, y=alto_tmp, hue=df_raw["class"],
    alpha=0.65, ax=axes[1, 0], palette="Set1"
)
axes[1, 0].set_title("3. Dimensiones de las Cajas Delimitadoras (px)")
axes[1, 0].set_xlabel("Ancho (px)")
axes[1, 0].set_ylabel("Alto (px)")

# 4. Mapa de calor espacial del centro de las hojas
cx_tmp = (df_raw["xmin"] + df_raw["xmax"]) / 2.0
cy_tmp = (df_raw["ymin"] + df_raw["ymax"]) / 2.0
hb = axes[1, 1].hist2d(cx_tmp, cy_tmp, bins=16, range=[[0, 512], [0, 512]], cmap="YlGn")
axes[1, 1].invert_yaxis()
axes[1, 1].set_title("4. Densidad Espacial de Centros de Hoja (512x512)")
axes[1, 1].set_xlabel("Coordenada X (px)")
axes[1, 1].set_ylabel("Coordenada Y (px)")
plt.colorbar(hb[3], ax=axes[1, 1], label="Frecuencia")

plt.subplots_adjust(hspace=0.42, wspace=0.25)
plt.show()

---
## Unidad 2 · Preparación de Datos e Ingeniería de Características (Sin Fuga de Datos)
En esta etapa aplicamos las reglas de depuración detectadas en el EDA:
1. **Deduplicación y normalización:** Eliminamos 1 registro duplicado exacto, unificamos la etiqueta inconsistente `Unhealthy` $\rightarrow$ `unhealthy tobacco leaf` y aislamos las 5 cajas `Unlabeled` para obtener **705 registros válidos** (639 sanas = 90,64 %; 66 enfermas = 9,36 %).
2. **Ingeniería de variables (12 predictores geométricos, espaciales y de contexto):** Calculamos dimensiones (`bbox_ancho_px`, `bbox_alto_px`, `bbox_area_px2`, `perimetro_px`), cobertura relativa (`ratio_cobertura_img` %), forma (`relacion_aspecto`, `compacidad_bbox`), ubicación (`centro_x_norm`, `centro_y_norm`, `dist_centro_img`) y contexto de escena (`hojas_en_imagen`, `toca_borde`).
3. **Extracción visual opcional:** Si subiste la carpeta/ZIP con las imágenes, el script extrae adicionalmente RGB medio, saturación, brillo y textura Laplaciana de cada hoja recortada.
4. **Prevención de *Data Leakage*:** El escalador (`RobustScaler`) se ajusta **únicamente** dentro de `Pipeline` sobre el conjunto de entrenamiento tras el split estratificado 80/20.

In [5]:
df = df_raw.drop_duplicates().copy()
df["base_image"] = df["filename"].str.split("_jpg.rf.").str[0]
df["class_clean"] = df["class"].replace({"Unhealthy": "unhealthy tobacco leaf"})

# Separar cajas sin etiquetar
df_unlabeled = df[df["class_clean"] == "Unlabeled"].copy()
df_clean = df[df["class_clean"] != "Unlabeled"].copy()
df_clean["target_enferma"] = (df_clean["class_clean"] == "unhealthy tobacco leaf").astype(int)

# Variables de contexto de escena y geometría
leaves_per_img = df.groupby("filename")["class"].transform("count")
df_clean["hojas_en_imagen"] = leaves_per_img.loc[df_clean.index]
df_clean["bbox_ancho_px"] = df_clean["xmax"] - df_clean["xmin"]
df_clean["bbox_alto_px"] = df_clean["ymax"] - df_clean["ymin"]
df_clean["bbox_area_px2"] = df_clean["bbox_ancho_px"] * df_clean["bbox_alto_px"]
df_clean["ratio_cobertura_img"] = (df_clean["bbox_area_px2"] / (512.0 * 512.0)) * 100.0
df_clean["perimetro_px"] = 2 * (df_clean["bbox_ancho_px"] + df_clean["bbox_alto_px"])
df_clean["relacion_aspecto"] = df_clean["bbox_ancho_px"] / df_clean["bbox_alto_px"]
df_clean["compacidad_bbox"] = (4 * np.pi * df_clean["bbox_area_px2"]) / (df_clean["perimetro_px"] ** 2)
df_clean["centro_x_norm"] = ((df_clean["xmin"] + df_clean["xmax"]) / 2.0) / 512.0
df_clean["centro_y_norm"] = ((df_clean["ymin"] + df_clean["ymax"]) / 2.0) / 512.0
df_clean["dist_centro_img"] = np.sqrt((df_clean["centro_x_norm"] - 0.5) ** 2 + (df_clean["centro_y_norm"] - 0.5) ** 2)
df_clean["toca_borde"] = (
    (df_clean["xmin"] <= 2) | (df_clean["ymin"] <= 2) |
    (df_clean["xmax"] >= 510) | (df_clean["ymax"] >= 510)
).astype(int)

features_geom = [
    "bbox_ancho_px", "bbox_alto_px", "bbox_area_px2", "ratio_cobertura_img",
    "perimetro_px", "relacion_aspecto", "compacidad_bbox",
    "centro_x_norm", "centro_y_norm", "dist_centro_img",
    "hojas_en_imagen", "toca_borde"
]

# Extracción automática de características visuales si el usuario subió las imágenes
features_visuales = []
if os.path.isdir(CARPETA_IMAGENES):
    try:
        import cv2
        mapa_imgs = {}
        for raiz, _, archivos in os.walk(CARPETA_IMAGENES):
            for arch in archivos:
                if arch.lower().endswith((".jpg", ".jpeg", ".png")):
                    mapa_imgs[arch] = os.path.join(raiz, arch)
        if mapa_imgs:
            vis_rows = {}
            for fn, grp in df_clean.groupby("filename"):
                if fn not in mapa_imgs:
                    continue
                img = cv2.imread(mapa_imgs[fn])
                if img is None:
                    continue
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
                gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                for idx, r in grp.iterrows():
                    rec_rgb = img_rgb[int(r["ymin"]):int(r["ymax"]), int(r["xmin"]):int(r["xmax"])]
                    rec_hsv = img_hsv[int(r["ymin"]):int(r["ymax"]), int(r["xmin"]):int(r["xmax"])]
                    rec_gr = gris[int(r["ymin"]):int(r["ymax"]), int(r["xmin"]):int(r["xmax"])]
                    if rec_rgb.size > 0:
                        vis_rows[idx] = {
                            "vis_R_medio": float(rec_rgb[:, :, 0].mean()),
                            "vis_G_medio": float(rec_rgb[:, :, 1].mean()),
                            "vis_B_medio": float(rec_rgb[:, :, 2].mean()),
                            "vis_saturacion": float(rec_hsv[:, :, 1].mean()),
                            "vis_brillo": float(rec_hsv[:, :, 2].mean()),
                            "vis_textura_lap": float(np.log1p(cv2.Laplacian(rec_gr, cv2.CV_64F).var()))
                        }
            if vis_rows:
                df_vis = pd.DataFrame.from_dict(vis_rows, orient="index")
                df_clean = df_clean.join(df_vis)
                for c in df_vis.columns:
                    df_clean[c] = df_clean[c].fillna(df_clean[c].median())
                features_visuales = list(df_vis.columns)
                print(f"Características visuales extraídas desde imágenes reales: {features_visuales}")
    except Exception as e:
        print("Aviso: se continúa solo con variables geométricas/espaciales:", e)

features_modelo = features_geom + features_visuales

print(f"Dataset depurado: {len(df_clean)} registros válidos ({len(df_unlabeled)} Unlabeled aislados).")
print("Distribución final de clases:")
print(df_clean["class_clean"].value_counts())
print("\nEstadísticos descriptivos de las 12 variables base:")
display(df_clean[features_geom].describe().T[["count", "mean", "std", "min", "50%", "max"]].round(3))

Dataset depurado: 705 registros válidos (5 Unlabeled aislados).
Distribución final de clases:
class_clean
tobacco leaf              639
unhealthy tobacco leaf     66
Name: count, dtype: int64

Estadísticos descriptivos de las 12 variables base:


,count,mean,std,min,50%,max
bbox_ancho_px,705.0,228.298,158.447,19.000,176.000,511.000
bbox_alto_px,705.0,230.165,138.180,23.000,206.000,511.000
bbox_area_px2,705.0,67304.089,76462.017,684.000,30178.000,259588.000
ratio_cobertura_img,705.0,25.674,29.168,0.261,11.512,99.025
perimetro_px,705.0,916.925,543.165,110.000,736.000,2038.000
relacion_aspecto,705.0,1.182,0.872,0.227,0.843,5.145
compacidad_bbox,705.0,0.708,0.076,0.428,0.731,0.785
centro_x_norm,705.0,0.489,0.208,0.028,0.499,0.973
centro_y_norm,705.0,0.512,0.212,0.037,0.501,0.963
dist_centro_img,705.0,0.256,0.151,0.002,0.270,0.654


---
## Unidad 3 · Clasificación Supervisada (Detección de Hoja Enferma)
Evaluamos 8 configuraciones de algoritmos sobre un split estratificado **80/20** (`564` entrenamiento / `141` prueba, con `13` hojas enfermas en prueba), comparando modelos lineales, basados en distancias (con y sin escalado `RobustScaler`), máquinas de soporte vectorial y ensambles de árboles.

In [6]:
X = df_clean[features_modelo]
y = df_clean["target_enferma"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]} muestras ({y_train.sum()} enfermas) | Test: {X_test.shape[0]} muestras ({y_test.sum()} enfermas)")

modelos_clf = {
    "Baseline (Dummy Mayoría)": DummyClassifier(strategy="most_frequent"),
    "KNN (Sin Escalar, k=5)": KNeighborsClassifier(n_neighbors=5),
    "KNN (RobustScaler, k=5)": Pipeline([("scaler", RobustScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))]),
    "Regresión Logística (Balanceada)": Pipeline([
        ("scaler", RobustScaler()),
        ("lr", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "SVM RBF (Balanceado)": Pipeline([
        ("scaler", RobustScaler()),
        ("svm", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE))
    ]),
    "Árbol de Decisión (Balanceado)": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest (Balanceado)": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE)
}

resultados_clf = []
predicciones_prob = {}

for nombre, modelo in modelos_clf.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1] if hasattr(modelo, "predict_proba") else y_pred
    predicciones_prob[nombre] = y_prob
    resultados_clf.append({
        "Modelo": nombre,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision (Enferma)": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall (Enferma)": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1 (Enferma)": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "F1 Macro": round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4)
    })

df_res_clf = pd.DataFrame(resultados_clf)
display(df_res_clf)

Train: 564 muestras (53 enfermas) | Test: 141 muestras (13 enfermas)


,Modelo,Accuracy,Precision (Enferma),Recall (Enferma),F1 (Enferma),F1 Macro,ROC-AUC
0,Baseline (Dummy Mayoría),0.9078,0.0000,0.0000,0.0000,0.4758,0.5000
1,"KNN (Sin Escalar, k=5)",0.9362,0.6667,0.6154,0.6400,0.8025,0.9769
2,"KNN (RobustScaler, k=5)",0.9362,0.7500,0.4615,0.5714,0.7685,0.9041
3,Regresión Logística (Balanceada),0.7163,0.1860,0.6154,0.2857,0.5544,0.7572
4,SVM RBF (Balanceado),0.8936,0.4583,0.8462,0.5946,0.7667,0.9657
5,Árbol de Decisión (Balanceado),0.9574,0.7059,0.9231,0.8000,0.8881,0.9486
6,Random Forest (Balanceado),0.9716,0.9091,0.7692,0.8333,0.9089,0.9940
7,Gradient Boosting,0.9858,0.9231,0.9231,0.9231,0.9576,0.9970


In [7]:
mejor_gb = modelos_clf["Gradient Boosting"]
rf_model = modelos_clf["Random Forest (Balanceado)"]
y_pred_gb = mejor_gb.predict(X_test)
cm_gb = confusion_matrix(y_test, y_pred_gb)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Matriz de confusión Gradient Boosting
sns.heatmap(
    cm_gb, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0, 0],
    xticklabels=["Sana (0)", "Enferma (1)"], yticklabels=["Sana (0)", "Enferma (1)"]
)
axes[0, 0].set_title("1. Matriz de Confusión — Gradient Boosting (Test 80/20)")
axes[0, 0].set_xlabel("Predicción")
axes[0, 0].set_ylabel("Clase Real")

# 2. Curvas ROC comparativas
for nombre in ["Regresión Logística (Balanceada)", "SVM RBF (Balanceado)", "Random Forest (Balanceado)", "Gradient Boosting"]:
    fpr, tpr, _ = roc_curve(y_test, predicciones_prob[nombre])
    auc_v = roc_auc_score(y_test, predicciones_prob[nombre])
    axes[0, 1].plot(fpr, tpr, lw=2, label=f"{nombre} (AUC={auc_v:.3f})")
axes[0, 1].plot([0, 1], [0, 1], "k--", lw=1, label="Azar")
axes[0, 1].set_title("2. Curvas ROC Comparativas")
axes[0, 1].set_xlabel("Tasa de Falsos Positivos (FPR)")
axes[0, 1].set_ylabel("Tasa de Verdaderos Positivos (Recall)")
axes[0, 1].legend(fontsize=8)

# 3. Curvas Precisión-Recall
for nombre in ["SVM RBF (Balanceado)", "Random Forest (Balanceado)", "Gradient Boosting"]:
    prec, rec, _ = precision_recall_curve(y_test, predicciones_prob[nombre])
    ap_v = average_precision_score(y_test, predicciones_prob[nombre])
    axes[1, 0].plot(rec, prec, lw=2, label=f"{nombre} (PR-AUC={ap_v:.3f})")
axes[1, 0].axhline(y_test.mean(), color="gray", linestyle="--", label="Prevalencia base (9.2%)")
axes[1, 0].set_title("3. Curvas Precisión–Recall (Clase Enferma)")
axes[1, 0].set_xlabel("Recall (Sensibilidad)")
axes[1, 0].set_ylabel("Precisión")
axes[1, 0].legend(fontsize=8)

# 4. Importancia de variables (Random Forest)
fi = pd.Series(rf_model.feature_importances_, index=features_modelo).sort_values(ascending=True)
fi.plot(kind="barh", ax=axes[1, 1], color="#2E8B57")
axes[1, 1].set_title("4. Importancia de Variables (Random Forest)")
axes[1, 1].set_xlabel("Importancia Relativa (Gini)")

plt.subplots_adjust(hspace=0.38, wspace=0.25)
plt.show()

---
## Unidad 4 · Regresión y Curva de Complejidad (Subajuste vs. Sobreajuste)
Modelamos la variable continua **`ratio_cobertura_img` (%)** (porcentaje del área del fotograma 512 × 512 px ocupado por cada hoja) a partir de su morfología, ubicación espacial, densidad de escena y estado sanitario, evaluando el compromiso sesgo-varianza a lo largo de distintas profundidades de árbol.

In [8]:
reg_features = [
    "relacion_aspecto", "compacidad_bbox", "dist_centro_img",
    "centro_x_norm", "centro_y_norm", "hojas_en_imagen", "toca_borde", "target_enferma"
]
Xr = df_clean[reg_features]
yr = df_clean["ratio_cobertura_img"]

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.20, random_state=RANDOM_STATE)

modelos_reg = {
    "Regresión Lineal Múltiple": Pipeline([("scaler", RobustScaler()), ("lr", LinearRegression())]),
    "Regresión Ridge (alpha=1.0)": Pipeline([("scaler", RobustScaler()), ("ridge", Ridge(alpha=1.0))]),
    "Árbol Regresor (depth=2 - Subajuste)": DecisionTreeRegressor(max_depth=2, random_state=RANDOM_STATE),
    "Árbol Regresor (depth=5 - Equilibrio)": DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE),
    "Árbol Regresor (sin límite - Sobreajuste)": DecisionTreeRegressor(max_depth=None, random_state=RANDOM_STATE),
    "Random Forest Regressor (200 árboles)": RandomForestRegressor(n_estimators=200, max_depth=8, random_state=RANDOM_STATE)
}

res_reg = []
for nombre, reg in modelos_reg.items():
    reg.fit(Xr_tr, yr_tr)
    p_tr = reg.predict(Xr_tr)
    p_te = reg.predict(Xr_te)
    res_reg.append({
        "Modelo Regresor": nombre,
        "R2 Train": round(r2_score(yr_tr, p_tr), 4),
        "R2 Test": round(r2_score(yr_te, p_te), 4),
        "MAE Test (%)": round(mean_absolute_error(yr_te, p_te), 2),
        "RMSE Test (%)": round(np.sqrt(mean_squared_error(yr_te, p_te)), 2)
    })

display(pd.DataFrame(res_reg))

# Curva de complejidad (profundidad de árbol 1 a 15)
depths = list(range(1, 16))
r2_tr_curve, r2_te_curve = [], []
for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE).fit(Xr_tr, yr_tr)
    r2_tr_curve.append(r2_score(yr_tr, dt.predict(Xr_tr)))
    r2_te_curve.append(r2_score(yr_te, dt.predict(Xr_te)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(depths, r2_tr_curve, "o-", label="R² Entrenamiento", color="#1F77B4")
axes[0].plot(depths, r2_te_curve, "s-", label="R² Prueba (Test)", color="#D62728")
axes[0].axvline(5, color="green", linestyle="--", label="Profundidad óptima (depth=5)")
axes[0].set_title("Curva de Complejidad: Subajuste vs. Sobreajuste")
axes[0].set_xlabel("Profundidad máxima del árbol (max_depth)")
axes[0].set_ylabel("Coeficiente R²")
axes[0].legend()

rf_reg = modelos_reg["Random Forest Regressor (200 árboles)"]
pred_rf = rf_reg.predict(Xr_te)
axes[1].scatter(yr_te, pred_rf, alpha=0.65, color="#2E8B57")
axes[1].plot([0, 100], [0, 100], "k--", lw=1.2)
axes[1].set_title(f"Real vs. Predicho — Random Forest (R²={r2_score(yr_te, pred_rf):.4f})")
axes[1].set_xlabel("Cobertura Foliar Real (%)")
axes[1].set_ylabel("Cobertura Foliar Predicha (%)")

plt.tight_layout()
plt.show()

,Modelo Regresor,R2 Train,R2 Test,MAE Test (%),RMSE Test (%)
0,Regresión Lineal Múltiple,0.8521,0.8045,9.49,12.14
1,Regresión Ridge (alpha=1.0),0.8521,0.8053,9.46,12.12
2,Árbol Regresor (depth=2 - Subajuste),0.8524,0.8571,7.39,10.38
3,Árbol Regresor (depth=5 - Equilibrio),0.9609,0.9266,4.97,7.44
4,Árbol Regresor (sin límite - Sobreajuste),1.0000,0.9661,2.29,5.06
5,Random Forest Regressor (200 árboles),0.9888,0.9666,3.49,5.02


---
## Unidad 5 · Aprendizaje No Supervisado: Clustering y Reglas de Asociación
1. **Segmentación por Clustering (`K-Means` y `Ward`):** Descubrimos perfiles naturales de captura y morfología foliar sin utilizar etiquetas de clase, evaluando el índice **Silhouette** y la concordancia entre algoritmos mediante **Adjusted Rand Index (ARI)**.
2. **Reglas de Asociación (`Apriori`):** Extraemos patrones frecuentes *Si–Entonces* entre condiciones geométricas, encuadre de cámara y estado sanitario, ordenados por **Lift**.

In [9]:
clust_features = ["ratio_cobertura_img", "relacion_aspecto", "compacidad_bbox", "dist_centro_img", "hojas_en_imagen"]
Xc = RobustScaler().fit_transform(df_clean[clust_features])

metricas_k = []
for k in [2, 3, 4, 5]:
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit(Xc)
    ward = AgglomerativeClustering(n_clusters=k, linkage="ward").fit(Xc)
    metricas_k.append({
        "k": k,
        "Inercia": round(km.inertia_, 1),
        "Silhouette": round(silhouette_score(Xc, km.labels_), 4),
        "ARI (K-Means vs Ward)": round(adjusted_rand_score(km.labels_, ward.labels_), 4)
    })

print("=== EVALUACIÓN DE NÚMERO DE CLÚSTERES ===")
display(pd.DataFrame(metricas_k))

# Segmentación final con k=3
km3 = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE).fit(Xc)
df_clean["cluster_k3"] = km3.labels_
perfiles = df_clean.groupby("cluster_k3")[clust_features + ["target_enferma", "toca_borde"]].mean().round(3)
perfiles["cantidad_hojas"] = df_clean["cluster_k3"].value_counts().sort_index()
print("\n=== PERFILES DE LOS 3 CLÚSTERES DESCUBIERTOS ===")
display(perfiles)

# Minería de Reglas de Asociación (Apriori exacto sobre ítems binarios)
tx = pd.DataFrame({
    "Enferma": df_clean["target_enferma"] == 1,
    "Sana": df_clean["target_enferma"] == 0,
    "Escena_Multicapa": df_clean["hojas_en_imagen"] > 1,
    "Hoja_Unica": df_clean["hojas_en_imagen"] == 1,
    "Toca_Borde": df_clean["toca_borde"] == 1,
    "Sin_Recorte_Borde": df_clean["toca_borde"] == 0,
    "Encuadre_Central": df_clean["dist_centro_img"] <= 0.25,
    "Cobertura_Alta": df_clean["ratio_cobertura_img"] >= 25.0,
    "Cobertura_Baja": df_clean["ratio_cobertura_img"] < 12.0
})

def extraer_reglas_apriori(df_bin, min_support=0.05, min_conf=0.65, min_lift=1.30):
    arr = df_bin.to_numpy(dtype=bool)
    cols = list(df_bin.columns)
    n = len(arr)
    soportes = {}
    for size in [1, 2, 3]:
        for comb in itertools.combinations(range(len(cols)), size):
            sup = arr[:, comb].all(axis=1).mean()
            if sup >= min_support:
                soportes[comb] = sup
    filas = []
    for itemset, sup_xy in soportes.items():
        if len(itemset) < 2:
            continue
        for r in range(1, len(itemset)):
            for ant in itertools.combinations(itemset, r):
                cons = tuple(x for x in itemset if x not in ant)
                sup_x = soportes.get(ant)
                sup_y = soportes.get(cons)
                if sup_x and sup_y:
                    conf = sup_xy / sup_x
                    lift = conf / sup_y
                    if conf >= min_conf and lift >= min_lift:
                        filas.append({
                            "Antecedente (Si)": " + ".join(cols[i] for i in ant),
                            "Consecuente (Entonces)": " + ".join(cols[i] for i in cons),
                            "Soporte": round(sup_xy, 4),
                            "Confianza": round(conf, 4),
                            "Lift": round(lift, 4)
                        })
    return pd.DataFrame(filas).sort_values("Lift", ascending=False).drop_duplicates().head(8)

reglas_top = extraer_reglas_apriori(tx)
print("\n=== TOP REGLAS DE ASOCIACIÓN (APRIORI) ===")
display(reglas_top)

# Visualización PCA de los 3 clústeres
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(Xc)
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    x=coords[:, 0], y=coords[:, 1], hue=df_clean["cluster_k3"].astype(str),
    palette=["#E67E22", "#2E8B57", "#3465A4"], alpha=0.75, ax=ax
)
ax.set_title("Segmentación de Hojas en 3 Perfiles (Proyección PCA 2D)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)")
ax.legend(title="Clúster")
plt.tight_layout()
plt.show()

=== EVALUACIÓN DE NÚMERO DE CLÚSTERES ===

=== PERFILES DE LOS 3 CLÚSTERES DESCUBIERTOS ===

=== TOP REGLAS DE ASOCIACIÓN (APRIORI) ===


,k,Inercia,Silhouette,ARI (K-Means vs Ward)
0,2,1356.4,0.4769,0.9492
1,3,796.2,0.4979,0.8632
2,4,637.3,0.3939,0.6235
3,5,538.3,0.3903,0.7033


,ratio_cobertura_img,relacion_aspecto,compacidad_bbox,dist_centro_img,hojas_en_imagen,target_enferma,toca_borde,cantidad_hojas
cluster_k3,,,,,,,,
0,10.432,2.721,0.623,0.328,5.892,0.092,0.569,130
1,77.587,1.150,0.770,0.074,1.405,0.039,0.987,153
2,11.549,0.719,0.712,0.300,7.303,0.114,0.552,422


,Antecedente (Si),Consecuente (Entonces),Soporte,Confianza,Lift
29,Toca_Borde + Encuadre_Central,Hoja_Unica,0.1872,0.7952,4.2470
26,Hoja_Unica,Toca_Borde + Encuadre_Central,0.1872,1.0000,4.2470
33,Toca_Borde + Cobertura_Alta,Hoja_Unica,0.1872,0.7765,4.1471
30,Hoja_Unica,Toca_Borde + Cobertura_Alta,0.1872,1.0000,4.1471
16,Sana + Cobertura_Alta,Hoja_Unica,0.1872,0.7416,3.9607
14,Hoja_Unica,Sana + Cobertura_Alta,0.1872,1.0000,3.9607
34,Hoja_Unica,Encuadre_Central + Cobertura_Alta,0.1872,1.0000,3.5787
37,Encuadre_Central + Cobertura_Alta,Hoja_Unica,0.1872,0.6701,3.5787


---
## Unidad 6 · Validación Cruzada, Auditoría de Fuga por *Data Augmentation* y Equidad
En datasets exportados con *data augmentation* (donde cada foto original genera hasta 3 versiones espejadas), una validación cruzada estándar (`StratifiedKFold`) reparte variantes de una misma hoja entre entrenamiento y prueba, inflando artificialmente las métricas. Comparamos ese esquema frente a **`StratifiedGroupKFold`** agrupado por `base_image`, y auditamos el rendimiento por subgrupos de encuadre.

In [10]:
gb_cv = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE)
groups = df_clean["base_image"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
sgkf = StratifiedGroupKFold(n_splits=5)

cv_std = cross_validate(gb_cv, X, y, cv=skf, scoring=["accuracy", "precision", "recall", "f1", "roc_auc"])
cv_grp = cross_validate(gb_cv, X, y, groups=groups, cv=sgkf, scoring=["accuracy", "precision", "recall", "f1", "roc_auc"])

comp_cv = []
for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    comp_cv.append({
        "Métrica": m.upper(),
        "StratifiedKFold (Estándar)": f"{cv_std[f'test_{m}'].mean():.4f} ± {cv_std[f'test_{m}'].std():.4f}",
        "StratifiedGroupKFold (Por Foto Base)": f"{cv_grp[f'test_{m}'].mean():.4f} ± {cv_grp[f'test_{m}'].std():.4f}"
    })

print("=== AUDITORÍA DE FUGA DE DATOS POR AUGMENTATION (5-FOLD CV) ===")
display(pd.DataFrame(comp_cv))

# Auditoría de equidad por subgrupos en Test 80/20
audit = X_test.copy()
audit["y_true"] = y_test
audit["y_pred"] = y_pred_gb

subgrupos = []
for val, sub in audit.groupby("toca_borde"):
    etq = "Hoja Completa (Sin Borde)" if val == 0 else "Hoja Cortada en Borde"
    subgrupos.append({
        "Criterio": "Encuadre",
        "Subgrupo": etq,
        "N Test": len(sub),
        "Casos Enfermas": int(sub["y_true"].sum()),
        "Accuracy": round(accuracy_score(sub["y_true"], sub["y_pred"]), 4),
        "Recall (Sensibilidad)": round(recall_score(sub["y_true"], sub["y_pred"], zero_division=0), 4)
    })

audit["tipo_escena"] = np.where(audit["hojas_en_imagen"] == 1, "Hoja Individual (1)", "Racimo Multicapa (>1)")
for val, sub in audit.groupby("tipo_escena"):
    subgrupos.append({
        "Criterio": "Densidad de Escena",
        "Subgrupo": val,
        "N Test": len(sub),
        "Casos Enfermas": int(sub["y_true"].sum()),
        "Accuracy": round(accuracy_score(sub["y_true"], sub["y_pred"]), 4),
        "Recall (Sensibilidad)": round(recall_score(sub["y_true"], sub["y_pred"], zero_division=0), 4)
    })

print("\n=== AUDITORÍA DE EQUIDAD POR SUBGRUPO ===")
display(pd.DataFrame(subgrupos))

=== AUDITORÍA DE FUGA DE DATOS POR AUGMENTATION (5-FOLD CV) ===

=== AUDITORÍA DE EQUIDAD POR SUBGRUPO ===


,Métrica,StratifiedKFold (Estándar),StratifiedGroupKFold (Por Foto Base)
0,ACCURACY,0.9716 ± 0.0090,0.9020 ± 0.0192
1,PRECISION,0.9833 ± 0.0333,0.2817 ± 0.2423
2,RECALL,0.7143 ± 0.1036,0.2600 ± 0.3762
3,F1,0.8217 ± 0.0640,0.2384 ± 0.2771
4,ROC_AUC,0.9316 ± 0.0641,0.7770 ± 0.1573


,Criterio,Subgrupo,N Test,Casos Enfermas,Accuracy,Recall (Sensibilidad)
0,Encuadre,Hoja Completa (Sin Borde),48,6,0.9792,1.0000
1,Encuadre,Hoja Cortada en Borde,93,7,0.9892,0.8571
2,Densidad de Escena,Hoja Individual (1),21,0,1.0000,0.0000
3,Densidad de Escena,Racimo Multicapa (>1),120,13,0.9833,0.9231
